In [0]:
%sql
-- 1. Metadata ke liye naya schema
CREATE SCHEMA IF NOT EXISTS vstone_catalog.metadata_schema;

-- 2. Dedicated External Volume sirf Checkpoints ke liye
CREATE VOLUME IF NOT EXISTS vstone_catalog.metadata_schema.autoloader_checkpoints;

-- 3. Purani table ko saaf karein taaki fresh start ho
DROP TABLE IF EXISTS vstone_catalog.bronze.listings_json_autoloader;

#JSON Ingestion (Chunk 3) via Auto Loader

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# 1. NEW UNIQUE ISOLATED PATHS
input_path = "/Volumes/vstone_catalog/raw/chunks/"

# Checkpoint aur Schema ko ekdum unique folder mein rakhein
checkpoint_root = "/Volumes/vstone_catalog/raw/checkpoints/autoloader_vFinal_Isolated"
schema_path = f"{checkpoint_root}/schema"
checkpoint_path = f"{checkpoint_root}/checkpoint"

# 2. METADATA CLEANUP
try:
    dbutils.fs.rm(checkpoint_root, True)
    print(f"🧹 Isolated location cleaned: {checkpoint_root}")
except Exception as e:
    print(f"⚠️ Cleanup note: {str(e)}")

# 3. AUTO LOADER: Pure String Ingestion
df_json = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path) # Fresh Isolation
    .option("pathGlobFilter", "1_main_chunk_3.json")
    .option("multiLine", "true") 
    .option("cloudFiles.inferColumnTypes", "false") # Sab data string
    .load(input_path)
    .withColumn("load_dt", current_timestamp())
    .withColumn("source_file", lit("1_main_chunk_3.json")))

# 4. WRITE STREAM
query = (df_json.writeStream
    .option("checkpointLocation", checkpoint_path) # Fresh Isolation
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("vstone_catalog.bronze.listings_json_autoloader"))

query.awaitTermination()
print("✅ JSON Ingestion Complete: Isolated paths fixed the overlap!")

In [0]:
%sql
 select * from vstone_catalog.bronze.listings_json_autoloader limit 5;